<a href="https://colab.research.google.com/github/opendatas2017/NMC/blob/main/NMCourse_fast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ускорение


In [1]:
import numpy as np
import sys

print("Numpy version is ",np.__version__)  # Должно показать версию, например 2.0.2

print("Python version is ",sys.version)

Numpy version is  2.0.2
Python version is  3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


Инсталяция codon  - интерактивная: отвечаем y

In [2]:
!/bin/bash -c "$(curl -fsSL https://exaloop.io/install.sh)"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
codon-deploy-linux-x86_64/python/
codon-deploy-linux-x86_64/python/MANIFEST.in
codon-deploy-linux-x86_64/python/README.md
codon-deploy-linux-x86_64/python/setup.py
codon-deploy-linux-x86_64/python/codon/
codon-deploy-linux-x86_64/python/codon/version.py
codon-deploy-linux-x86_64/python/codon/jit.pxd
codon-deploy-linux-x86_64/python/codon/__init__.py
codon-deploy-linux-x86_64/python/codon/decorator.py
codon-deploy-linux-x86_64/python/codon/jit.pyx
codon-deploy-linux-x86_64/python/codon_jit.egg-info/
codon-deploy-linux-x86_64/python/codon_jit.egg-info/dependency_links.txt
codon-deploy-linux-x86_64/python/codon_jit.egg-info/requires.txt
codon-deploy-linux-x86_64/python/codon_jit

In [4]:
!pip install codon-jit -q

In [5]:
!pip install line_profiler -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.8 MB/s eta 0:00:00


Измерение скорости

In [6]:
import numpy as np
import timeit
SIZE: int = 2_000_000
# Сравним скорость: список vs массив с timeit
print(f"🏎️  БЕНЧМАРК: Список vs NumPy ({SIZE} элементов)")

# 1. Сначала NumPy (быстрый)
arr_data = np.arange(SIZE)

def numpy_sum() -> np.float64:
    return np.sum(arr_data * 2)

arr_time = timeit.timeit(numpy_sum, number=100)
print(f"NumPy:      {arr_time:.3f} сек")

# 2. Python список (медленный)
list_data = list(range(SIZE))
def list_sum() -> float:
    return sum(list_data * 2)

list_time = timeit.timeit(list_sum, number=100)
print(f"Список:     {list_time:.3f} сек (×{list_time/arr_time:.0f} медленнее)")

print(f"\n🚀 NumPy {list_time/arr_time:.0f} раз быстрее!")


🏎️  БЕНЧМАРК: Список vs NumPy (2000000 элементов)
NumPy:      0.983 сек
Список:     12.068 сек (×12 медленнее)

🚀 NumPy 12 раз быстрее!


### 💡 Вывод
**NumPy работает в десятки раз быстрее** стандартных списков Python.  

**Причины:**  
1. **Векторизация** — операции выполняются над всем массивом сразу на уровне C, а не поэлементно в цикле Python.  
2. **Эффективная память** — данные хранятся компактно и непрерывно, что ускоряет доступ процессора.  

👉 *Правило для численных методов: всегда используйте массивы `np.array` вместо списков `list`.*

In [7]:
import timeit
import random
import numpy as np

# Размер выборки (100 тысяч элементов)
SIZE = 100_000

# Подготовка данных
data_list = [random.random() for _ in range(SIZE)]
data_array = np.array(data_list)  # Конвертируем в массив NumPy заранее

# --- 1. Явный цикл Python (Самый медленный) ---
# Максимальные накладные расходы интерпретатора на каждой итерации
def slow_sum(xs):
    s = 0.0
    for x in xs:
        s += x
    return s

# --- 2. Встроенная функция sum (Быстрее) ---
# Реализована на C, но работает с объектами Python
def fast_sum(xs):
    return sum(xs)

# --- 3. NumPy sum (Самый быстрый) ---
# Векторизованная операция, работа с непрерывной памятью, SIMD-инструкции
def numpy_sum(xs):
    return np.sum(xs)

# --- Замер времени ---
# Выполняем каждую функцию 100 раз
slow_time = timeit.timeit(lambda: slow_sum(data_list), number=100)
fast_time = timeit.timeit(lambda: fast_sum(data_list), number=100)
numpy_time = timeit.timeit(lambda: numpy_sum(data_array), number=100)

# --- Вывод результатов ---
print(f"Явный цикл (Python): {slow_time:.4f} сек")
print(f"Встроенная sum:      {fast_time:.4f} сек (×{slow_time/fast_time:.1f} быстрее цикла)")
print(f"NumPy sum:           {numpy_time:.4f} сек (×{slow_time/numpy_time:.1f} быстрее цикла)")

Явный цикл (Python): 0.2822 сек
Встроенная sum:      0.0999 сек (×2.8 быстрее цикла)
NumPy sum:           0.0039 сек (×72.0 быстрее цикла)


### 💡 Вывод
**Иерархия производительности:**  
`Цикл for` ≪ `Встроенная sum()` < `NumPy np.sum()`

**Почему:**  
1. **Цикл:** Интерпретатор Python обрабатывает каждую строку кода отдельно (медленно).  
2. **Встроенная:** Логика вынесена в C, но данные всё ещё объекты Python.  
3. **NumPy:** Данные хранятся плотно (primitive types), вычисления векторизованы и используют возможности процессора (SIMD).


в colab есть магические команды , можно сделать так

In [8]:
%timeit -n 100 -r 10 slow_sum(data_list)


3.99 ms ± 949 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


### 💡 Вывод для запоминания
**%timeit — стандартный инструмент профилирования в Jupyter.**

**Параметры:**  
| Параметр | Значение | Зачем нужен |
|----------|----------|-------------|
| `-n` | число повторений кода | Усредняет шум внутри одного прогона |
| `-r` | число независимых прогонов | Позволяет оценить стабильность (± отклонение) |

**Что смотреть в выводе:**  
`12.3 ms ± 456 µs per loop` → среднее время выполнения одной итерации с погрешностью.

👉 *Используйте `%timeit` для быстрой проверки гипотез о производительности прямо в ноутбуке.*

##  Профилирование кода: Поиск узких мест с cProfile

In [9]:
import math
import time
import cProfile
import pstats

# --- Функция 1: Вычислительная нагрузка (CPU-bound) ---
# Много математических операций, нагружает процессор
def heavy_compute(n):
    s = 0.0
    for i in range(n):
        s += math.sqrt(i) * math.sin(i)
    return s

# --- Функция 2: Имитация задержки (I/O-bound) ---
# Ожидание (например, чтение файла, сеть), процессор простаивает
def sleepy(n):
    time.sleep(n / 1000)  # Задержка в секундах

# --- Основная функция ---
def main():
    heavy_compute(300_000)  # Тяжёлые вычисления
    sleepy(100)             # Имитация ожидания (100 мс)


with cProfile.Profile() as pr:
        main()

    # Обработка и вывод статистики
stats = pstats.Stats(pr)
    # Сортируем по собственному времени функции (без учёта вызовов внутри)
stats.sort_stats("tottime").print_stats(10)

         600528 function calls (600520 primitive calls) in 0.315 seconds

   Ordered by: internal time
   List reduced from 109 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.100    0.100    0.100    0.100 {built-in method time.sleep}
        1    0.099    0.099    0.191    0.191 /tmp/ipykernel_4647/2811392108.py:8(heavy_compute)
   300000    0.054    0.000    0.054    0.000 {built-in method math.sin}
   300000    0.044    0.000    0.044    0.000 {built-in method math.sqrt}
        2    0.006    0.003    0.006    0.003 /usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py:219(<lambda>)
        4    0.005    0.001    0.006    0.002 /usr/lib/python3.12/asyncio/events.py:86(_run)
        1    0.005    0.005    0.008    0.008 {method 'send' of '_socket.socket' objects}
       14    0.000    0.000    0.000    0.000 /usr/local/lib/python3.12/dist-packages/zmq/sugar/socket.py:632(send)
        1    0.000    0.000  

### 💡 Вывод
**cProfile — встроенный профилировщик Python для поиска «узких мест».**

**Ключевые метрики в отчёте:**  
| Метрика | Что показывает | Зачем нужно |
|---------|----------------|-------------|
| `tottime` | Время в функции (без вызовов других) | Найти самую «тяжёлую» функцию |
| `cumtime` | Полное время (включая вызовы внутри) | Понять общее влияние на программу |
| `ncalls` | Количество вызовов | Выявить чрезмерные повторения |

**Практический смысл:**  
- `heavy_compute` — нагружает **CPU** (математика) → можно ускорить через NumPy  
- `sleepy` — **I/O-ожидание** → можно ускорить через асинхронность  

👉 *Сначала профилируйте, потом оптимизируйте. Не гадайте, где медленно — измеряйте!*

##  профилирование: LineProfiler

In [10]:
from line_profiler import LineProfiler
%load_ext line_profiler

In [11]:
def heavy_compute(n):
    s = 0.0
    for i in range(n):
        a = math.sqrt(i)
        b = math.sin(i)
        s += a * b
    return s

In [12]:
lp = LineProfiler()
lp.add_function(heavy_compute)
lp_wrapper = lp(heavy_compute)

lp_wrapper(300_000)
lp.print_stats()

Timer unit: 1e-09 s

Total time: 0.57613 s
File: /tmp/ipykernel_4647/997312074.py
Function: heavy_compute at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def heavy_compute(n):
     2         1       8971.0   8971.0      0.0      s = 0.0
     3    300001  129949184.0    433.2     22.6      for i in range(n):
     4    300000  154935706.0    516.5     26.9          a = math.sqrt(i)
     5    300000  165938183.0    553.1     28.8          b = math.sin(i)
     6    300000  125291959.0    417.6     21.7          s += a * b
     7         1       6143.0   6143.0      0.0      return s



In [15]:
%lprun -f heavy_compute heavy_compute(300_000)

### 💡 Вывод
**LineProfiler показывает время выполнения КАЖДОЙ строки кода.**

**В отличие от cProfile:**  
| Инструмент | Уровень детализации | Когда использовать |
|------------|---------------------|-------------------|
| `cProfile` | По функциям | Быстрый обзор, какие функции тормозят |
| `LineProfiler` | По строкам | Глубокий анализ внутри «тяжёлой» функции |

**Что смотреть в отчёте:**  
- **Line #** — номер строки  
- **Hits** — сколько раз выполнена  
- **Time per hit (µs)** — среднее время одного выполнения  
- **% Time** — доля от общего времени функции  

👉 *Используйте LineProfiler, когда нашли «тяжёлую» функцию через cProfile и хотите понять, какая именно строка внутри неё тормозит.*

### Оптимизация рекурсии: Кэширование (Memoization)

In [18]:
from functools import lru_cache
import math
import time

# --- 1. "Тяжёлая" функция с кэшем ---
#  cache сохраняет результаты вызовов.
# При повторном вызове с тем же аргументом возвращает значение из памяти, а не вычисляет.
@lru_cache(maxsize=None)
def f(x: float) -> float:
    # Имитация сложной вычислительной задачи
    return math.exp(-x**2) * math.sin(10 * x) + math.cos(3 * x)

# --- 2. Адаптивное интегрирование (рекурсивное) ---
def adaptive_trapezoid(a, b, eps=1e-6, depth=0, max_depth=50):
    """
    Рекурсивная схема трапеций.
    Важная особенность: граничные точки подынтервалов совпадают.
    Например, mid текущего интервала станет левой границой для правого подынтервала.
    Без кэша функция f() вычислялась бы многократно для одних и тех же точек.
    """
    mid = 0.5 * (a + b)

    # Вычисление значений функции в ключевых точках
    # Благодаря кэшу, повторные вызовы f(a), f(b), f(mid) мгновенны
    fa, fb, fm = f(a), f(b), f(mid)

    # Оценка интеграла: 1 трапеция vs 2 трапеции
    I_big = (b - a) * (fa + fb) / 2
    I_left = (mid - a) * (fa + fm) / 2
    I_right = (b - mid) * (fm + fb) / 2
    I_small = I_left + I_right

    # Условие остановки: достигнута точность или глубина рекурсии
    if depth >= max_depth or abs(I_small - I_big) < eps:
        return I_small

    # Рекурсивный шаг
    return (adaptive_trapezoid(a, mid, eps, depth + 1, max_depth) +
            adaptive_trapezoid(mid, b, eps, depth + 1, max_depth))

# --- 3. Запуск и анализ ---
a, b = 0.0, 1.0
f.cache_clear()  # Очищаем кэш перед новым замером

t0 = time.perf_counter()
I = adaptive_trapezoid(a, b, eps=1e-8)
t1 = time.perf_counter()

print(f"Интеграл I ≈ {I:.8f}")
print(f"Время: {t1 - t0:.6f} сек")

# Статистика кэша: hits (попадания) vs misses (промахи)
print("Статистика кэша:", f.cache_info())

Интеграл I ≈ 0.18343617
Время: 0.002505 сек
Статистика кэша: CacheInfo(hits=3652, misses=1829, maxsize=None, currsize=1829)


misses = сколько раз функция реально считалась;

hits = сколько раз мы просто взяли значение из кэша, избежав повторного численного вычисления;





### 💡 Вывод
**Кэширование (`lru_cache`)  важно для рекурсивных численных методов.**

**Почему это работает:**  
В адаптивных схемах соседние интервалы имеют **общие граничные точки**.  
- **Без кэша:** `f(x)` вычисляется заново для каждой границы на каждом уровне рекурсии (экспоненциальный рост вызовов).  
- **С кэшем:** `f(x)` вычисляется один раз, последующие обращения занимают ~O(1) времени.

**Что показывает `cache_info()`:**  
- `hits`: сколько раз результат взят из памяти (экономия времени).  
- `misses`: сколько раз функция вычислена реально.  

👉 *Правило: Если рекурсивная функция вызывается с повторяющимися аргументами — используйте `@lru_cache`.*

In [19]:
import os
from time import perf_counter
import time
import timeit
import numpy as np
from concurrent.futures import ProcessPoolExecutor


# ==============================
# Базовая функция: подсчёт попаданий в круг (векторизованная)
# ==============================

def count_hits_block_numpy(block, n_strata: int, samples_per_stratum: int) -> tuple[int, int]:
    """
    Обрабатывает блок страт: генерирует точки и считает попадания в круг.
    Возвращает (попадания, всего_точек) для этого блока.
    """
    if not block:
        return 0, 0

    h = 1.0 / n_strata  # Размер одной страты
    block = np.array(block, dtype=np.int64)  # Преобразуем список индексов в массив

    K = block.shape[0]  # Количество страт в блоке

    # Генерация случайных точек внутри каждой страты
    # shape: (K, samples_per_stratum) — матрица точек для всех страт блока
    u = np.random.random(size=(K, samples_per_stratum))
    v = np.random.random(size=(K, samples_per_stratum))

    # Координаты левых нижних углов страт
    x0 = (block[:, 0] * h).reshape(K, 1)   # по оси X
    y0 = (block[:, 1] * h).reshape(K, 1)   # по оси Y

    # Сдвиг точек внутрь страты
    x = x0 + u * h
    y = y0 + v * h

    # Проверка условия попадания в единичный круг: x² + y² ≤ 1
    inside = (x * x + y * y) <= 1.0
    hits = int(inside.sum())  # Суммируем все попадания
    total = K * samples_per_stratum  # Общее число точек в блоке

    return hits, total


# ==============================
# 1) Последовательная версия (один процесс)
# ==============================

def estimate_pi_stratified_numpy(n_samples: int, n_strata: int = 100) -> float:
    """
    Стратифицированный Монте-Карло для оценки π.
    Все вычисления выполняются последовательно в одном процессе.
    """
    n_strata2 = n_strata * n_strata  # Общее число страт (сетка)
    samples_per_stratum = n_samples // n_strata2

    if samples_per_stratum == 0:
        raise ValueError("Слишком мало точек на страту — увеличьте n_samples")

    # Создаём список всех страт: (i, j) — индексы по осям
    all_strata = [(i, j) for i in range(n_strata) for j in range(n_strata)]

    # Обрабатываем все страты сразу в одной функции
    hits, total = count_hits_block_numpy(all_strata, n_strata, samples_per_stratum)
    return 4.0 * hits / total  # Формула оценки π


# ==============================
# 2) Параллельная версия (пул процессов)
# ==============================

def _worker_block(block, n_strata: int, samples_per_stratum: int) -> tuple[int, int]:
    """
    Функция-воркер для запуска в отдельном процессе.
    Просто вызывает векторизованную функцию подсчёта.
    """
    return count_hits_block_numpy(block, n_strata, samples_per_stratum)


def estimate_pi_stratified_process_pool_numpy(n_samples: int, n_strata: int = 100) -> float:
    """
    Параллельная версия: страты делятся на блоки,
    каждый блок обрабатывается в отдельном процессе.
    """
    n_strata2 = n_strata * n_strata
    samples_per_stratum = n_samples // n_strata2

    if samples_per_stratum == 0:
        raise ValueError("Слишком мало точек на страту — увеличьте n_samples")

    all_strata = [(i, j) for i in range(n_strata) for j in range(n_strata)]

    # Определяем число рабочих процессов (в Colab обычно 2 CPU)
    n_workers = os.cpu_count() or 4

    # Делим страты на блоки по числу процессов
    chunk_size = (len(all_strata) + n_workers - 1) // n_workers
    blocks = [all_strata[k:k + chunk_size] for k in range(0, len(all_strata), chunk_size)]

    hits_total = 0
    total_total = 0

    # Запускаем пул процессов
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        # Отправляем задачи в пул
        futures = [
            pool.submit(_worker_block, block, n_strata, samples_per_stratum)
            for block in blocks
        ]
        # Собираем результаты по мере готовности
        for f in futures:
            h, t = f.result()
            hits_total += h
            total_total += t

    return 4.0 * hits_total / total_total


# ==============================
# 3) Замер времени и сравнение
# ==============================

if __name__ == "__main__":
    n_samples = 20_000_000
    n_strata = 100

    print(f"📊 Параметры: n_samples = {n_samples:,}, n_strata = {n_strata}")
    print(f"💻 Доступно CPU: {os.cpu_count()}")

    # --- Последовательный запуск ---
    t0 = perf_counter()
    pi_single = estimate_pi_stratified_numpy(n_samples, n_strata)
    t1 = perf_counter()
    print(f"\n[Один процесс] π ≈ {pi_single:.6f}, время = {t1 - t0:.3f} сек")

    # --- Параллельный запуск ---
    t0 = perf_counter()
    pi_proc = estimate_pi_stratified_process_pool_numpy(n_samples, n_strata)
    t1 = perf_counter()
    print(f"[Пул процессов] π ≈ {pi_proc:.6f}, время = {t1 - t0:.3f} сек")

    # --- Статистика через timeit (более надёжно) ---
    print(f"\n🔁 Усреднённый замер (5 прогонов × 8 итераций):")

    first_time = timeit.repeat(
        lambda: estimate_pi_stratified_numpy(n_samples, n_strata),
        repeat=5, number=8
    )
    sec_time = timeit.repeat(
        lambda: estimate_pi_stratified_process_pool_numpy(n_samples, n_strata),
        repeat=5, number=8
    )

    print(f"Один процесс:  {np.mean(first_time):.3f} сек ± {np.std(first_time):.3f}")
    print(f"Пул процессов: {np.mean(sec_time):.3f} сек ± {np.std(sec_time):.3f}")

    speedup = np.mean(first_time) / np.mean(sec_time)
    print(f"\n🚀 Ускорение: ~{speedup:.2f}x")

📊 Параметры: n_samples = 20,000,000, n_strata = 100
💻 Доступно CPU: 2

[Один процесс] π ≈ 3.141589, время = 1.984 сек
[Пул процессов] π ≈ 3.141530, время = 1.644 сек

🔁 Усреднённый замер (5 прогонов × 8 итераций):
Один процесс:  7.897 сек ± 0.465
Пул процессов: 6.796 сек ± 0.358

🚀 Ускорение: ~1.16x


### 💡 Вывод для запоминания
**Параллелизм ускоряет CPU-задачи, но есть нюансы:**

| Фактор | Влияние |
|--------|---------|
| ✅ Векторизация внутри процесса | Даёт основной прирост (NumPy на C) |
| ✅ Распределение по ядрам | Дополнительное ускорение ×N_cores |
| ❌ Накладные расходы | Создание процессов + копирование данных (IPC) |
| ❌ Ограничение Colab | Всего 2 CPU  |

**Почему в Colab ускорение скромное?**  
1. Всего **2 виртуальных ядра** → негде распараллелить.  
2. **Overhead** на запуск процессов и передачу данных может «съедать» выгоду на малых задачах.  


👉 *Правило: Параллельте только тяжёлые, независимые вычисления. Всегда сравнивайте с последовательной версией — иногда векторизация важнее многопоточности.*

In [20]:
# Чистый Python
def sum_squares_py(n: int) -> int:
    s = 0
    for i in range(n):
        s += i * i
    return s

# Та же функция с JIT (Numba)
from numba import njit  # эквивалент @jit(nopython=True) в новых версиях

@njit
def sum_squares_numba(n: int) -> int:
    s = 0
    for i in range(n):
        s += i * i
    return s

print(sum_squares_py(10**3))
# первый вызов: компиляция + выполнение
print(sum_squares_numba(10**3))


332833500
332833500


### 💡 Вывод
**Numba (@njit) превращает медленный цикл Python в быстрый машинный код.**

**Как это работает:**  
| Этап | Что происходит | Время |
|------|----------------|-------|
| 1-й вызов | Компиляция: анализ типов + генерация машинного кода | ~10-100 мс (один раз) |
| 2-й+ вызов | Выполнение скомпилированного кода | ~в 10-100× быстрее Python |

**Важно:**  
- ✅ Работает с числами (`int`, `float`), массивами `np.array`, циклами  
- ❌ Не поддерживает сложную логику Python (списки, словари, классы) в режиме `nopython`  

👉 *Правило: Если есть цикл по числам — оберните в `@njit`. Первый вызов для "прогрева", потом замеряйте скорость.*

In [21]:
%timeit -n 50 -r 100 sum_squares_numba(1000)

The slowest run took 61.39 times longer than the fastest. This could mean that an intermediate result is being cached.
372 ns ± 1.27 µs per loop (mean ± std. dev. of 100 runs, 50 loops each)


In [22]:
%timeit -n 50 -r 100 sum_squares_py(1000)

73.8 µs ± 13.2 µs per loop (mean ± std. dev. of 100 runs, 50 loops each)


### 💡 Вывод
**Numba даёт ускорение в 10–100 раз для численных циклов.**

**Типичные результаты:**  
| Реализация | Время на итерацию | Относительная скорость |
|------------|------------------|------------------------|
| `@njit` (Numba) | ~50–200 нс | **1×** (база) |
| Чистый Python | ~2–10 мкс | **×20–100 медленнее** |

**Ключевые моменты:**  
1. 🔥 **Всегда "прогревайте"** функцию с `@njit` перед `%timeit` — иначе замерите время компиляции, а не выполнения.  
2. 📊 **Параметры `-n` и `-r`** дают надёжную статистику: среднее ± отклонение.  
3. 🎯 **Эффект максимален** для циклов с арифметикой, где нет сложных Python-объектов.

👉 *Правило: `@njit` + "прогрев" = честный бенчмарк. Для численных задач это обязательный инструмент оптимизации.*

In [23]:
import numpy as np
from numba import njit

# Базовый вариант на NumPy + Python‑цикл
def transform_py(x: np.ndarray) -> np.ndarray:
    out = np.empty_like(x)
    for i in range(x.shape[0]):
        out[i] = x[i] - np.sqrt(x[i])
    return out

def transform_numpy(x: np.ndarray) -> np.ndarray:
    return x - np.sqrt(x)


# JIT‑вариант
@njit
def transform_numba(x: np.ndarray) -> np.ndarray:
    return x - np.sqrt(x)

x = np.arange(1000000, dtype=np.float64)
y = transform_py(x)
yn = transform_numba(x)
yn = transform_numba(x)  # 1-2 вызов компилирует функцию


In [24]:
%timeit -n 1 -r 10 transform_py(x)

1.57 s ± 328 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


In [25]:
# numpy

%timeit -n 50 -r 10 transform_numpy(x)

# Numba: та же логика, но внутри JIT
%timeit -n 50 -r 10 transform_numba(x)

3.96 ms ± 174 µs per loop (mean ± std. dev. of 10 runs, 50 loops each)
1.31 ms ± 65.7 µs per loop (mean ± std. dev. of 10 runs, 50 loops each)


In [ ]:
import codon
from numba import njit
import timeit

def is_prime_python(n):
    if n <= 1:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

@njit
def is_prime_numba(n):
    if n <= 1:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

@codon.jit
def is_prime_codon(n):
    if n <= 1:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

start, stop = 100_000, 200_000

# разогрев JIT (Numba и Codon), чтобы компиляция не попала в бенчмарк
_ = sum(1 for i in range(10_000, 10_100) if is_prime_numba(i))
_ = sum(1 for i in range(10_000, 10_100) if is_prime_codon(i))

NUMBER = 1      # сколько раз выполняем «большой» проход
REPEAT = 5      # сколько раз повторяем измерение

def bench(stmt, setup):
    times = timeit.repeat(stmt=stmt, setup=setup,
                          number=NUMBER, repeat=REPEAT)
    best = min(times)
    return best / NUMBER

t_py = bench(
    "sum(1 for i in range(start, stop) if is_prime_python(i))",
    "from __main__ import is_prime_python, start, stop"
)

t_numba = bench(
    "sum(1 for i in range(start, stop) if is_prime_numba(i))",
    "from __main__ import is_prime_numba, start, stop"
)

t_codon = bench(
    "sum(1 for i in range(start, stop) if is_prime_codon(i))",
    "from __main__ import is_prime_codon, start, stop"
)

print(f"[python] {t_py:.3f} s per run")
print(f"[numba ] {t_numba:.3f} s per run")
print(f"[codon ] {t_codon:.3f} s per run")


### 💡 Вывод
**Сравнение JIT-компиляторов для численных задач:**

| Компилятор | Ускорение | Плюсы | Минусы |
|------------|-----------|-------|--------|
| **Python** | 1× (база) | Универсальный, все библиотеки | Медленный для циклов |
| **Numba** | ×20–100 | Лёгкая интеграция, поддержка NumPy | Ограниченное подмножество Python |
| **Codon** | ×50–200 | Полная компиляция, без GIL | Менее зрелый, требует установки |

**Важные нюансы:**  
1. ⚠️ **Codon** недоступен в Colab — требует отдельной установки (`pip install codon`)  

2. 🔥 **"Прогрев" обязателен** для любого JIT — первый вызов включает компиляцию  

👉 *Правило: Numba — стандарт для научных вычислений. Codon — перспективная альтернатива для CPU-задач, где нужна максимальная скорость.*

### Максимальное ускорение: Компиляция всего скрипта через Codon

In [ ]:
# %%writefile — магия Jupyter: сохраняет код в отдельный файл
# Это нужно, потому что Codon компилирует файлы, а не ячейки ноутбука
%%writefile trap.py
import numpy as np
from time import perf_counter

# --- Подынтегральная функция ---
# Вычисляем sin(x) для массива точек (векторизованная операция)
def f(x: np.ndarray) -> np.ndarray:
    return np.sin(x)

# --- Метод трапеций для численного интегрирования ---
# Формула: ∫f(x)dx ≈ h * (0.5*y₀ + y₁ + ... + yₙ₋₁ + 0.5*yₙ)
def trap_numpy(a: float, b: float, n: int) -> float:
    # Создаём равномерную сетку из n+1 точки на отрезке [a, b]
    x = np.linspace(a, b, n + 1)

    # Вычисляем значения функции во всех точках сразу (векторизация)
    y = f(x)

    # Шаг сетки
    h = (b - a) / n

    # Применяем формулу трапеций:
    # - крайние точки (y[0], y[-1]) с весом 0.5
    # - внутренние точки (y[1:-1]) с весом 1.0
    return h * (0.5 * y[0] + y[1:-1].sum() + 0.5 * y[-1])

# --- Точка входа: замер времени выполнения ---
def main():
    a, b = 0.0, np.pi      # Отрезок интегрирования [0, π]
    n = 10_000_000         # Число разбиений (10 миллионов!)

    t0 = perf_counter()    # Старт таймера
    res = trap_numpy(a, b, n)
    t1 = perf_counter()    # Финиш таймера

    print(f"Интеграл ≈ {res:.10f} (точное значение = 2.0)")
    print(f"Время    = {t1 - t0:.6f} сек")
    print(f"Ошибка   = {abs(res - 2.0):.2e}")

if __name__ == "__main__":
    main()

Overwriting trap.py


In [ ]:
!python trap.py


Интеграл ≈ 2.0000000000 (точное значение = 2.0)
Время    = 0.164854 сек
Ошибка   = 1.73e-14


In [ ]:
!/root/.codon/bin/codon run -release trap.py

Интеграл ≈ 2.0000000000 (точное значение = 2.0)
Время    = 0.065099 сек
Ошибка   = 1.64e-14


### 💡 Вывод
**Codon: когда нужно выжать максимум из численного кода**

| Критерий | Python + NumPy | Codon (AOT-компиляция) |
|----------|---------------|------------------------|
| **Скорость** | Быстро (векторизация) | Ещё быстрее (нативный код) |
| **Запуск** | Мгновенно | + время компиляции (один раз) |
| **GIL** | Есть (ограничение потоков) | Нет (полный параллелизм) |
| **Установка** | Везде есть | Требует отдельной настройки |

**Почему Codon может быть быстрее:**  
1. 🔓 **Нет GIL** — можно использовать многопоточность по-настоящему  
2. 🗑️ **Нет интерпретатора** — код выполняется напрямую процессором  


**Но есть нюанс:**  
Для векторизованных операций на NumPy разница может быть **не такой большой**, потому что тяжёлую работу уже делает оптимизированный C-код внутри NumPy.  

👉 *Правило: Сначала оптимизируйте алгоритм и используйте NumPy. Если этого мало — подключайте компиляторы типа Codon для финального рывка.*

In [ ]:
# %%writefile — сохраняем Cython-код в отдельный файл
# Расширение .pyx обязательно для Cython
%%writefile sum_loop.pyx
# cython: language_level=3

# cpdef — функция доступна и из Python, и из C (быстрее чем def)
cpdef double sum_loop(int n):
    # cdef — объявление переменных типа C (статическая типизация)
    # Это ключевое отличие от Python: типы известны на этапе компиляции
    cdef int i
    cdef double s = 0.0

    # Цикл выполняется на скорости C, без накладных расходов Python
    for i in range(n):
        s += i * i  # Прямая арифметика над машинными числами

    return s

Overwriting sum_loop.pyx


In [ ]:
# --- Файл сборки: setup.py ---
# Нужен для компиляции .pyx в расширяемый модуль (.so/.pyd)
%%writefile setup.py
from setuptools import setup
from Cython.Build import cythonize

setup(
    # cythonize преобразует .pyx в C-код, затем компилирует в бинарный модуль
    ext_modules=cythonize("sum_loop.pyx", compiler_directives={'language_level': "3"})
)

Overwriting setup.py


In [ ]:
!python setup.py build_ext --inplace


Compiling sum_loop.pyx because it changed.
[1/1] Cythonizing sum_loop.pyx
running build_ext
building 'sum_loop' extension
x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/include/python3.12 -c sum_loop.c -o build/temp.linux-x86_64-cpython-312/sum_loop.o
x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/sum_loop.o -L/usr/lib/x86_64-linux-gnu -o build/lib.linux-x86_64-cpython-312/sum_loop.cpython-312-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-312/sum_loop.cpython-312-x86_64-linux-gnu.so -> 


In [ ]:
import sum_loop

def sum_loop_python(n):
    s = 0.0
    for i in range(n):
        s += i * i
    return s


n = 10_000_000

result = sum_loop.sum_loop(n)


slow_time  = timeit.timeit(lambda: sum_loop_python(n), number=5)


fast_time =  timeit.timeit(lambda: sum_loop.sum_loop(n), number=5)

print(f"slow_sum total: {slow_time:.4f} s")
print(f"fast_sum total: {fast_time:.4f} s")
print(f"speedup ~ {slow_time / fast_time:.2f}x")


slow_sum total: 3.8430 s
fast_sum total: 0.0551 s
speedup ~ 69.77x


### 💡 Вывод
**Cython — мост между Python и C для максимального ускорения циклов**

| Критерий | Python | Cython |
|----------|--------|--------|
| **Типизация** | Динамическая (медленно) | Статическая `cdef` (быстро) |
| **Циклы** | Интерпретатор на каждом шаге | Машинный код C |
| **Скорость** | 1× (база) | ×50–200 быстрее |
| **Сложность** | Минимальная | Требуется компиляция |

**Ключевые директивы Cython:**  
- `cpdef` — функция вызывается и из Python, и из C (быстрее `def`)  
- `cdef` — переменные типа C (нет накладных расходов Python-объектов)  


**Когда использовать:**  
✅ Тяжёлые циклы с арифметикой  
✅ Код, который нельзя векторизовать через NumPy  
✅ Библиотеки для распространения (скомпилированный код)  

👉 *Правило: Если Numba не подходит (сложные типы, зависимости) — Cython даст сравнимое ускорение с большим контролем.*